# 02. Items Silver Layer Refining
This notebook focuses on the transformation and cleaning of the **Order Items** dataset. 
The goal is to transition the data from the **Raw** layer to the **Silver** layer by ensuring:
1. **Schema Integrity:** Converting data types into their appropriate formats.
2. **Referential Integrity:** Validating links between items, orders, products, and sellers.
3. **Business Logic Validation:** Checking price consistency and shipping timelines.

### 1. Data Ingestion
We start by loading the raw items dataset from the CSV source in the landing zone.

In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [3]:
# 1. Read the order items dataset in Delta format from Bronze layer in MinIO
df_items = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/order_items/")
)

# 2. Display the first 10 rows inside the notebook
display(df_items.limit(10))

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double, _ingested_at: timestamp, _source_file: string]

### 2. Data Quality Assessment: Duplicates Check
Before applying any transformations, we must identify the extent of data duplication. Removing duplicates at this early stage ensures that downstream processing (like Joins and Aggregations) is efficient and that our metrics are not artificially inflated.

In [4]:
total_count = df_items.count()
distinct_count = df_items.distinct().count()

duplicate_count = total_count - distinct_count
print(f"Total Duplicates: {duplicate_count}")

Total Duplicates: 0


### 3. Data Quality Assessment: Null Values Analysis
In this step, we perform a comprehensive scan across all columns to identify missing values (Nulls). Understanding the distribution of Nulls is critical for deciding our cleaning strategy—whether to drop rows, impute values, or move incomplete records to a quarantine table for further investigation.

In [5]:
from pyspark.sql import functions as F

# Calculate the count of Null values for each column in the dataset
null_counts = df_items.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_items.columns])
display(null_counts)

DataFrame[order_id: bigint, order_item_id: bigint, product_id: bigint, seller_id: bigint, shipping_limit_date: bigint, price: bigint, freight_value: bigint, _ingested_at: bigint, _source_file: bigint]

### 4. Reference Data Loading (Silver Orders)
To ensure **Referential Integrity**, we load the already refined `orders_silver` table. This dataset will act as our "Source of Truth" to validate that every item belongs to a legitimate and successfully processed order.

In [6]:
# Read the refined orders table from the Silver layer on MinIO
orders_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/orders/")
)

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Orphaned Records Imputation & Audit
Detect items missing valid parent orders, log them for audit, and re-map them to a '-1' placeholder to preserve data integrity.

In [7]:
# Perform a Left Anti Join to find records in df_items that do not have a matching order_id in orders_silver
# These "Orphaned" records are considered inconsistent and must be isolated
orphaned_items = df_items.join(orders_silver, "order_id", "left_anti")

# Log the counts for audit and monitoring purposes
print(f"Total Raw Items: {df_items.count()}")
print(f"Orphaned Items : {orphaned_items.count()}")

from pyspark.sql import functions as F

# ==========================================
# IDENTIFY ORPHANED ITEMS (PRE-AUDITING)
# ==========================================
# We use a Left Anti Join to capture item rows whose order_id does NOT exist in silver_orders
df_orphaned_items = df_items.join(orders_silver, "order_id", "left_anti")

# Count the orphaned items before updating them
orphaned_count = df_orphaned_items.count()
print(f"Data Quality Check: Found {orphaned_count} orphaned Item records.")

# ==========================================
# BUILD THE MASTER ITEM ERRORS AUDIT SYSTEM
# ==========================================
# Instead of a Quarantine table, we log these rows into a comprehensive Audit DataFrame with full context
if orphaned_count > 0:
    item_errors_audit = df_orphaned_items \
        .withColumn("error_reason", F.lit("Orphaned item: parent order_id was missing in Silver Orders - Imputed with -1"))
    print("🚩 Success: Orphans captured in the item_errors_audit DataFrame.")
else:
    # If no errors, create an empty schema-aligned DataFrame for the audit pipeline stability
    item_errors_audit = spark.createDataFrame([], schema=df_items.schema) \
        .withColumn("error_reason", F.lit("").cast("string"))

# ==========================================
# APPLY THE '-1' FALLBACK IMPUTATION FOR INTEGRITY
# ==========================================
# 3.1 Collect the exact orphaned order IDs to use for identification
# (Or we can check directly against the orders_silver lookup dynamically)
orders_lookup = orders_silver.select("order_id").distinct()

# 3.2 Update df_items: If an item's order_id is NOT in orders_silver, convert it to "-1"
# This safely routes the item to our global Unknown Order row instead of throwing it away
df_items = df_items.join(orders_lookup, "order_id", "left_outer") \
    .withColumn(
        "order_id",
        F.when(orders_lookup.order_id.isNotNull(), F.col("order_id")).otherwise(F.lit("-1"))
    ) \
    .drop(orders_lookup.order_id) # Drop the temporary lookup column after the operation

# ==========================================
# PIPELINE VALIDATION & METRICS PRINTING
# ==========================================
print("\n=== Final Items Pipeline Validation ===")
print(f"Total processed rows in df_items: {df_items.count()}")
print(f"Total rows successfully mapped to '-1': {df_items.filter(F.col('order_id') == '-1').count()}")

# Final Schema Check
df_items.printSchema()

Total Raw Items: 112650
Orphaned Items : 0
Data Quality Check: Found 0 orphaned Item records.

=== Final Items Pipeline Validation ===
Total processed rows in df_items: 112650
Total rows successfully mapped to '-1': 0
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



### 7. Cross-Reference Data Ingestion (Products & Sellers)
To achieve full **Referential Integrity**, we must ensure that every item refers to an existing product and a registered seller. In this step, we load the raw datasets for Products and Sellers to use them as lookup references for validation.

In [8]:
# 1. Load the Products dataset from the Bronze layer in MinIO
# Delta format automatically preserves exact column data types (no inferSchema needed)
products_raw = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/products/")
)

# 2. Load the Sellers dataset from the Bronze layer in MinIO
# This will be used to validate that each item is linked to a valid seller ID
sellers_raw = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/sellers/")
)

### 8. Integrity Validation: Missing Products and Sellers
We perform a **Left Anti Join** to identify any items referencing product or seller IDs that do not exist in their respective master tables. This ensures the dataset maintains strict relational consistency.

In [9]:
from pyspark.sql import functions as F

# ==========================================
# STEP 1: IDENTIFY PRODUCT & SELLER ORPHANS (FOR AUDITING)
# ==========================================
# Capture raw rows before imputation for the audit trail
product_orphans_df = df_items.join(products_raw, "product_id", "left_anti")
seller_orphans_df = df_items.join(sellers_raw, "seller_id", "left_anti")

product_orphans_count = product_orphans_df.count()
seller_orphans_count = seller_orphans_df.count()

print(f"Data Quality Check: Found {product_orphans_count} items with missing Product reference.")
print(f"Data Quality Check: Found {seller_orphans_count} items with missing Seller reference.")

# ==========================================
# STEP 2: APPEND TO THE MASTER AUDIT LOG SYSTEM
# ==========================================
# Log Product errors if they exist
if product_orphans_count > 0:
    product_audit = product_orphans_df.withColumn(
        "error_reason", 
        F.lit("Orphaned item: product_id not found in Products - Imputed with -1")
    )
    item_errors_audit = item_errors_audit.unionByName(product_audit, allowMissingColumns=True)

# Log Seller errors if they exist
if seller_orphans_count > 0:
    seller_audit = seller_orphans_df.withColumn(
        "error_reason", 
        F.lit("Orphaned item: seller_id not found in Sellers - Imputed with -1")
    )
    item_errors_audit = item_errors_audit.unionByName(seller_audit, allowMissingColumns=True)

print("🚩 Success: All identified orphans have been logged into the Audit system.")

# ==========================================
# STEP 3: APPLY THE '-1' FALLBACK IMPUTATION FOR INTEGRITY
# ==========================================
# 3.1 Impute Product IDs: If product_id is not in products_raw, replace with "-1"
products_lookup = products_raw.select("product_id").distinct()
df_items = df_items.join(products_lookup, "product_id", "left_outer") \
    .withColumn(
        "product_id",
        F.when(products_lookup.product_id.isNotNull(), F.col("product_id")).otherwise(F.lit("-1"))
    ) \
    .drop(products_lookup.product_id)

# 3.2 Impute Seller IDs: If seller_id is not in sellers_raw, replace with "-1"
sellers_lookup = sellers_raw.select("seller_id").distinct()
df_items = df_items.join(sellers_lookup, "seller_id", "left_outer") \
    .withColumn(
        "seller_id",
        F.when(sellers_lookup.seller_id.isNotNull(), F.col("seller_id")).otherwise(F.lit("-1"))
    ) \
    .drop(sellers_lookup.seller_id)

# ==========================================
# STEP 4: PIPELINE VALIDATION & PREVIEW
# ==========================================
print("\n=== Post-Imputation Verification ===")
# [FIXED HERE] Swapped inner double quotes with single quotes inside curly braces
print(f"Items mapped to Product '-1': {df_items.filter(F.col('product_id') == '-1').count()}")
print(f"Items mapped to Seller '-1': {df_items.filter(F.col('seller_id') == '-1').count()}")

# Show a sample to confirm the dataframe is intact and data types are healthy
df_items.select("order_id", "product_id", "seller_id").show(5)

Data Quality Check: Found 0 items with missing Product reference.
Data Quality Check: Found 0 items with missing Seller reference.
🚩 Success: All identified orphans have been logged into the Audit system.

=== Post-Imputation Verification ===
Items mapped to Product '-1': 0
Items mapped to Seller '-1': 0
+--------------------+--------------------+--------------------+
|            order_id|          product_id|           seller_id|
+--------------------+--------------------+--------------------+
|00010242fe8c5a6d1...|4244733e06e7ecb49...|48436dade18ac8b2b...|
|00018f77f2f0320c5...|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|
|000229ec398224ef6...|c777355d18b72b67a...|5b51032eddd242adc...|
|00024acbcdf0a6daa...|7634da152a4610f15...|9d7a1d34a50524090...|
|00042b26cf59d7ce6...|ac6c3623068f30de0...|df560393f3a51e745...|
+--------------------+--------------------+--------------------+
only showing top 5 rows



In [10]:
df_items.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Enforcement
Cast columns to target data types to ensure strict schema compliance for the `silver_order_items` table.

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType, DecimalType

# ==========================================
# STRICT TYPE CASTING FOR SILVER_ORDER_ITEMS
# ==========================================
# Casting every single column explicitly to ensure perfect schema compliance

df_items = df_items \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("order_item_id", F.col("order_item_id").cast(IntegerType())) \
    .withColumn("product_id", F.col("product_id").cast(StringType())) \
    .withColumn("seller_id", F.col("seller_id").cast(StringType())) \
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast(TimestampType())) \
    .withColumn("price", F.col("price").cast(DecimalType(10, 2))) \
    .withColumn("freight_value", F.col("freight_value").cast(DecimalType(10, 2)))

# ==========================================
# SCHEMA VERIFICATION
# ==========================================
print("=== FINAL SILVER ITEMS SCHEMA VALIDATION ===")
df_items.printSchema()

# Preview a sample to make sure decimal points are perfectly formatted (.xx)
df_items.select("order_id", "price", "freight_value").show(5)

=== FINAL SILVER ITEMS SCHEMA VALIDATION ===
root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+--------------------+------+-------------+
|            order_id| price|freight_value|
+--------------------+------+-------------+
|00010242fe8c5a6d1...| 58.90|        13.29|
|00018f77f2f0320c5...|239.90|        19.93|
|000229ec398224ef6...|199.00|        17.87|
|00024acbcdf0a6daa...| 12.99|        12.79|
|00042b26cf59d7ce6...|199.90|        18.14|
+--------------------+------+-------------+
only showing top 5 rows



### 9. Statistical Data Profiling
We use the `describe()` function to generate summary statistics for all numerical and string columns. This helps in detecting potential anomalies, such as negative prices, zero values, or unexpected date ranges, providing a final health check before proceeding.

In [12]:
display(df_items.describe())

DataFrame[summary: string, seller_id: string, product_id: string, order_id: string, order_item_id: string, price: string, freight_value: string, _source_file: string]

In [13]:
df_items.select(
"price","freight_value"
).show(10)

+------+-------------+
| price|freight_value|
+------+-------------+
| 58.90|        13.29|
|239.90|        19.93|
|199.00|        17.87|
| 12.99|        12.79|
|199.90|        18.14|
| 21.90|        12.69|
| 19.90|        11.85|
|810.00|        70.75|
|145.95|        11.65|
| 53.99|        11.40|
+------+-------------+
only showing top 10 rows



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data-Driven Chronology Repair
Calculate historical shipping buffers to impute missing or logically invalid `shipping_limit_date` values, ensuring a consistent and valid item lifecycle.

In [14]:
from pyspark.sql import functions as F

# ==========================================
# STEP 1: INNER JOIN WITH ORDERS FOR TIMESTAMPS LOOKUP
# ==========================================
items_with_dates = df_items.join(
    orders_silver.select("order_id", "order_purchase_timestamp", "order_approved_at"),
    on="order_id",
    how="inner"
)

# ==========================================
# STEP 2: DYNAMICALLY CALCULATE THE MEDIAN SHIPPING BUFFER FROM VALID DATA
# ==========================================
valid_records = items_with_dates.filter(
    F.col("shipping_limit_date").isNotNull() & 
    F.col("order_approved_at").isNotNull() & 
    (F.col("shipping_limit_date") >= F.col("order_approved_at"))
)

valid_diffs = valid_records.withColumn(
    "valid_buffer_days", 
    F.datediff("shipping_limit_date", "order_approved_at")
)

median_row = valid_diffs.select(
    F.percentile_approx("valid_buffer_days", 0.5).alias("median_days")
).collect()

# [FIXED HERE] Swapped 'null' with 'None' to strictly comply with Python syntax
calculated_median_days = int(median_row[0]["median_days"]) if median_row and median_row[0]["median_days"] is not None else 4

print(f"📊 Data-Driven Insight: The historical Median shipping buffer calculated from valid records is: {calculated_median_days} days.")

# ==========================================
# STEP 3: DEFINE CHRONOLOGICAL ANOMALIES & NULLS DETECTOR
# ==========================================
is_limit_null = F.col("shipping_limit_date").isNull()
is_limit_invalid = (F.col("shipping_limit_date") < F.col("order_purchase_timestamp")) | \
                   (F.coalesce(F.col("shipping_limit_date") < F.col("order_approved_at"), F.lit(False)))

has_chronology_issue = is_limit_null | is_limit_invalid

# ==========================================
# STEP 4: CAPTURE AND APPEND TO THE MASTER AUDIT LOG
# ==========================================
items_chronology_errors = items_with_dates.filter(has_chronology_issue) \
    .withColumn(
        "error_reason", 
        F.when(is_limit_null, F.lit("Missing shipping_limit_date (Null) - Rebuilt using fallback data-driven logic"))
         .otherwise(F.lit(f"Shipping limit date is before purchase/approval date - Imputed using calculated median of {calculated_median_days} days"))
    )

if items_chronology_errors.count() > 0:
    item_errors_audit = item_errors_audit.unionByName(items_chronology_errors, allowMissingColumns=True)
    print(f"🚩 Success: Logged {items_chronology_errors.count()} timestamp anomalies into item_errors_audit.")

# ==========================================
# STEP 5: DYNAMIC RE-ORDERING & IMPUTATION LOGIC USING CALCULATED MEDIAN
# ==========================================
fallback_shipping_limit = F.coalesce(
    F.date_add(F.col("order_approved_at"), calculated_median_days),
    F.date_add(F.col("order_purchase_timestamp"), calculated_median_days)
)

items_refined = items_with_dates.withColumn(
    "shipping_limit_date",
    F.when(has_chronology_issue, fallback_shipping_limit).otherwise(F.col("shipping_limit_date"))
)

# ==========================================
# STEP 6: CLEANUP & FINAL VERIFICATION
# ==========================================
items_refined = items_refined.drop("order_purchase_timestamp", "order_approved_at")

print("\n=== SHIPPING LIMIT CHRONOLOGY REPAIR REPORT ===")
print(f"Final safe items count in items_refined (Zero Rows Deleted): {items_refined.count()}")
print(f"Any remaining missing shipping dates? {items_refined.filter(F.col('shipping_limit_date').isNull()).count()}")

items_refined.select("order_id", "order_item_id", "shipping_limit_date").show(5)

📊 Data-Driven Insight: The historical Median shipping buffer calculated from valid records is: 6 days.
🚩 Success: Logged 127 timestamp anomalies into item_errors_audit.

=== SHIPPING LIMIT CHRONOLOGY REPAIR REPORT ===
Final safe items count in items_refined (Zero Rows Deleted): 112650
Any remaining missing shipping dates? 0
+--------------------+-------------+-------------------+
|            order_id|order_item_id|shipping_limit_date|
+--------------------+-------------+-------------------+
|00010242fe8c5a6d1...|            1|2017-09-19 09:45:35|
|00018f77f2f0320c5...|            1|2017-05-03 11:05:13|
|000229ec398224ef6...|            1|2018-01-18 14:48:30|
|00024acbcdf0a6daa...|            1|2018-08-15 10:10:18|
|00042b26cf59d7ce6...|            1|2017-02-13 13:57:51|
+--------------------+-------------+-------------------+
only showing top 5 rows



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Seller Performance Analytics
Integrate shipping benchmarks with order lifecycle statuses to evaluate seller compliance and quantify handling variance.

In [15]:
from pyspark.sql import functions as F

# ==========================================
# 1. SELECT NECESSARY COLUMNS FROM ORDERS
# ==========================================
# [IMPROVED] We now pull order_status along with the carrier date to drive dynamic business logic
shipping_reference = orders_silver.select("order_id", "order_status", "order_delivered_carrier_date")

# ==========================================
# 2. JOIN WITH ITEMS TABLE
# ==========================================
items_joined = items_refined.join(shipping_reference, on="order_id", how="left")

# ==========================================
# 3. CALCULATE SELLER COMPLIANCE & PERFORMANCE
# ==========================================
items_final = items_joined.withColumn(
    "seller_handling_days", 
    F.datediff("order_delivered_carrier_date", "shipping_limit_date")
).withColumn(
    "seller_performance",
    # 3.1 Check business operations status first to clear up the "Unknowns"
    F.when(F.col("order_status") == "canceled", "Canceled")
     .when(F.col("order_status") == "unavailable", "Unavailable")
     .when(F.col("order_status").isin("shipped", "processing", "approved", "created"), "In Progress")
     
     # 3.2 If the order is successfully delivered, we evaluate the seller's alignment with the deadline
     .when(F.col("order_status") == "delivered", 
           F.when(F.col("seller_handling_days") < 0, "Early")
            .when(F.col("seller_handling_days") == 0, "On Time")
            .when(F.col("seller_handling_days") > 0, "Late Delivery to Carrier")
            .otherwise("Unfulfilled")
     )
     .otherwise("Unknown")
).withColumn(
    "abs_seller_handling",
    # [PROTECTED] Only calculate absolute variance if it's a delivered order with valid timestamps
    F.when((F.col("order_status") == "delivered") & (F.col("seller_handling_days").isNotNull()), 
           F.abs(F.col("seller_handling_days")))
     .otherwise(F.lit(None))
)

# ==========================================
# 4. PIPELINE VALIDATION AND AUDIT PREVIEW
# ==========================================
print("=== Seller Performance Breakdown ===")
items_final.groupBy("order_status", "seller_performance").count().show()

print("=== Previewing Sample Items Data ===")
items_final.select(
    "order_id", 
    "order_status", 
    "shipping_limit_date", 
    "order_delivered_carrier_date", 
    "seller_performance", 
    "abs_seller_handling"
).show(10)

=== Seller Performance Breakdown ===
+------------+--------------------+-----+
|order_status|  seller_performance|count|
+------------+--------------------+-----+
|   delivered|             On Time| 7542|
|   delivered|               Early|96093|
|     shipped|         In Progress| 1185|
|    invoiced|             Unknown|  359|
|    approved|         In Progress|    3|
|   delivered|Late Delivery to ...| 6562|
|    canceled|            Canceled|  542|
|  processing|         In Progress|  357|
| unavailable|         Unavailable|    7|
+------------+--------------------+-----+

=== Previewing Sample Items Data ===
+--------------------+------------+-------------------+----------------------------+--------------------+-------------------+
|            order_id|order_status|shipping_limit_date|order_delivered_carrier_date|  seller_performance|abs_seller_handling|
+--------------------+------------+-------------------+----------------------------+--------------------+-------------------+
|

In [16]:
items_final.select(
    "order_id", "shipping_limit_date", "order_delivered_carrier_date","seller_handling_days",
    "abs_seller_handling", "seller_performance"
).show(10)

+--------------------+-------------------+----------------------------+--------------------+-------------------+--------------------+
|            order_id|shipping_limit_date|order_delivered_carrier_date|seller_handling_days|abs_seller_handling|  seller_performance|
+--------------------+-------------------+----------------------------+--------------------+-------------------+--------------------+
|00010242fe8c5a6d1...|2017-09-19 09:45:35|         2017-09-19 18:34:16|                   0|                  0|             On Time|
|00018f77f2f0320c5...|2017-05-03 11:05:13|         2017-05-04 14:35:00|                   1|                  1|Late Delivery to ...|
|000229ec398224ef6...|2018-01-18 14:48:30|         2018-01-16 12:36:48|                  -2|                  2|               Early|
|00024acbcdf0a6daa...|2018-08-15 10:10:18|         2018-08-10 13:28:00|                  -5|                  5|               Early|
|00042b26cf59d7ce6...|2017-02-13 13:57:51|         2017-02-16 

### 14. Schema Finalization
To maintain a clean and normalized data structure, we drop the auxiliary columns borrowed from the Orders table. This prevents data redundancy in the Silver layer, as these delivery dates are already master-recorded in the Orders table.

In [17]:
# Drop columns used for calculation to maintain data normalization
items_final = items_final.drop("order_delivered_carrier_date")

display(items_final.limit(5))

DataFrame[order_id: string, seller_id: string, product_id: string, order_item_id: int, shipping_limit_date: timestamp, price: decimal(10,2), freight_value: decimal(10,2), _ingested_at: timestamp, _source_file: string, order_status: string, seller_handling_days: int, seller_performance: string, abs_seller_handling: int]

In [18]:
items_final.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- seller_handling_days: integer (nullable = true)
 |-- seller_performance: string (nullable = false)
 |-- abs_seller_handling: integer (nullable = true)



In [19]:
# Drop the ambiguous order_status column explicitly
items_final = items_final.drop("order_status")

print("✓ Success: 'order_status' column dropped successfully.")
print("=== Current items_final Schema ===")
items_final.printSchema()

✓ Success: 'order_status' column dropped successfully.
=== Current items_final Schema ===
root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- seller_handling_days: integer (nullable = true)
 |-- seller_performance: string (nullable = false)
 |-- abs_seller_handling: integer (nullable = true)



### 15. Seller Performance Logic Coverage Validation**
**Objective:** Validates the newly generated `seller_performance` column specifically to ensure it contains zero `NULL` values.

In [20]:
from pyspark.sql import functions as F

print("=== SELLER PERFORMANCE NULLS & DISTRIBUTION CHECK ===")

# Calculate total NULLs count in the performance column
null_count = items_final.filter(F.col("seller_performance").isNull()).count()
total_count = items_final.count()

print(f"Total Rows in Items: {total_count}")
print(f"Rows with NULL Performance: {null_count}")
print(f"Missing Performance Ratio: {(null_count / total_count) * 100:.2f}%")
print("="*50)

# Display the full distribution of performance values including NULLs
print("\n=== PERFORMANCE VALUE DISTRIBUTION ===")
items_final.groupBy("seller_performance") \
    .agg(F.count("*").alias("count")) \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)

=== SELLER PERFORMANCE NULLS & DISTRIBUTION CHECK ===
Total Rows in Items: 112650
Rows with NULL Performance: 0
Missing Performance Ratio: 0.00%

=== PERFORMANCE VALUE DISTRIBUTION ===
+------------------------+-----+
|seller_performance      |count|
+------------------------+-----+
|Early                   |96093|
|On Time                 |7542 |
|Late Delivery to Carrier|6562 |
|In Progress             |1545 |
|Canceled                |542  |
|Unknown                 |359  |
|Unavailable             |7    |
+------------------------+-----+



**Result & Insights:**
* **Perfect Coverage:** Missing performance records is exactly **0.00%** (0 rows), confirming the transformation logic handles all rows successfully.
* **Distribution Baseline:** The absolute majority of preparations fall under `Early` (**96093 items**), providing a highly positive performance baseline.

### 16. Investigative Join: Order Status Distribution**
**Objective:** Inspects the underlying `order_status` of the items by joining with `silver_orders` to check if incomplete or canceled orders exist in this pipeline.

In [21]:
from pyspark.sql import functions as F

print("=== ORDER STATUS DISTRIBUTION INSIDE ITEMS ===")

# Joined with orders and grouped by order_status using order_item_id
items_final.join(orders_silver, "order_id") \
    .groupBy("order_status") \
    .agg(
        F.count("order_item_id").alias("total_items"),
        F.countDistinct("order_id").alias("unique_orders")
    ) \
    .orderBy(F.col("total_items").desc()) \
    .show(truncate=False)

=== ORDER STATUS DISTRIBUTION INSIDE ITEMS ===
+------------+-----------+-------------+
|order_status|total_items|unique_orders|
+------------+-----------+-------------+
|delivered   |110197     |96478        |
|shipped     |1185       |1106         |
|canceled    |542        |461          |
|invoiced    |359        |312          |
|processing  |357        |301          |
|unavailable |7          |6            |
|approved    |3          |2            |
+------------+-----------+-------------+



### Order Status Analysis
Evaluation of item distribution across order lifecycle stages to identify fulfillment volumes and operational bottlenecks.

**Insights:**
* **Dominant Status:** The vast majority of items (**110,197**) are associated with `delivered` orders.
* **Exceptions:** A small fraction of items are currently in pending or problematic states (`shipped`, `canceled`, `invoiced`, `processing`), indicating high fulfillment efficiency.

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Final Schema Enforcement
Cast all columns to the defined target schema and align the final DataFrame to the specific production column set, ensuring data cleanliness and consistency.

In [22]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType, DecimalType

# ==========================================
# STEP 1: STRICT TYPE CASTING & SCHEMA ENFORCEMENT
# ==========================================
# We cast every single column explicitly to match your exact target data types

items_final = items_final \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("order_item_id", F.col("order_item_id").cast(IntegerType())) \
    .withColumn("product_id", F.col("product_id").cast(StringType())) \
    .withColumn("seller_id", F.col("seller_id").cast(StringType())) \
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast(TimestampType())) \
    .withColumn("price", F.col("price").cast(DecimalType(10, 2))) \
    .withColumn("freight_value", F.col("freight_value").cast(DecimalType(10, 2))) \
    .withColumn("seller_handling_days", F.col("seller_handling_days").cast(IntegerType())) \
    .withColumn("abs_seller_handling", F.col("abs_seller_handling").cast(IntegerType())) \
    .withColumn("seller_performance", F.col("seller_performance").cast(StringType()))

# ==========================================
# STEP 2: SELECT & ALIGN EXACTLY THE 10 TARGET COLUMNS
# ==========================================
# This drops temporary columns like 'order_status' and 'order_delivered_carrier_date' 
# that we only used for calculation, leaving the schema perfectly clean.

target_items_columns = [
    "order_id", 
    "order_item_id", 
    "product_id", 
    "seller_id", 
    "shipping_limit_date", 
    "price", 
    "freight_value", 
    "seller_handling_days", 
    "abs_seller_handling", 
    "seller_performance"
]

items_final = items_final.select(*target_items_columns)

# ==========================================
# STEP 3: FINAL SCHEMA VERIFICATION PRINTING
# ==========================================
print("=== FINAL SILVER ORDER ITEMS SCHEMA VALIDATION ===")
items_final.printSchema()

# Preview sample data to check numerical format formatting
print("=== Previewing Final Schema-Compliant Data ===")
items_final.select("order_id", "price", "seller_handling_days", "seller_performance").show(5)

=== FINAL SILVER ORDER ITEMS SCHEMA VALIDATION ===
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- seller_handling_days: integer (nullable = true)
 |-- abs_seller_handling: integer (nullable = true)
 |-- seller_performance: string (nullable = false)

=== Previewing Final Schema-Compliant Data ===
+--------------------+------+--------------------+--------------------+
|            order_id| price|seller_handling_days|  seller_performance|
+--------------------+------+--------------------+--------------------+
|00010242fe8c5a6d1...| 58.90|                   0|             On Time|
|00018f77f2f0320c5...|239.90|                   1|Late Delivery to ...|
|000229ec398224ef6...|199.00|                  -2|           

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Final Audit Persistence
Persist the item-level data quality audit log to the Lakehouse in Delta format, ensuring pipeline stability and historical traceability of anomalies.

In [23]:
# ==========================================
# FINAL PERSISTENCE: SAVING ITEM ERRORS AUDIT TABLE
# ==========================================
# Checking if the audit dataframe actually contains any records before writing
error_count = item_errors_audit.count()

print(f"Preparing to save Data Quality Audit Log... Total anomalies found: {error_count}")

# Define the absolute MinIO S3A storage path for the items audit logs
ITEMS_AUDIT_LOG_PATH = "s3a://silver/qa_issues/silver_items_errors_audit/"

# We save the audit dataframe using Delta format with overwrite mode for pipeline re-run safety (FIXED: Using direct path)
item_errors_audit.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(ITEMS_AUDIT_LOG_PATH)

# Explicitly refresh the Delta cache for this path to ensure instant data governance visibility
spark.catalog.refreshByPath(ITEMS_AUDIT_LOG_PATH)

print(f"Success: 'silver_items_errors_audit' has been safely persisted to: {ITEMS_AUDIT_LOG_PATH}")
print("Notebook Data Governance and Imputation Steps are now 100% Complete!")

Preparing to save Data Quality Audit Log... Total anomalies found: 127
Success: 'silver_items_errors_audit' has been safely persisted to: s3a://silver/qa_issues/silver_items_errors_audit/
Notebook Data Governance and Imputation Steps are now 100% Complete!


### 18. Final Data Persistence (Delta & Parquet)
The final stage of the Silver pipeline is to persist the refined data. We save the results in two formats:
1. **Delta Table:** For seamless integration within the Lakehouse and Metastore, enabling ACID transactions and versioning.
2. **Parquet File:** For high-performance storage and easy portability for downstream analytics or external team sharing.

In [26]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING REFINED SILVER ORDER ITEMS (CORRECTED)
# ==========================================================================

# 1. Define the target absolute storage paths on MinIO
SILVER_ITEMS_DELTA_PATH   = "s3a://silver/refined/order_items/"
SILVER_ITEMS_PARQUET_PATH = "s3a://silver/refined/order_items_parquet/"

# 2. Save as a Delta Table using direct MinIO S3A paths instead of saveAsTable
items_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_ITEMS_DELTA_PATH)

# Refresh the Delta cache for immediate query capability inside the cluster
spark.catalog.refreshByPath(SILVER_ITEMS_DELTA_PATH)


# 3. Export as Parquet files to the Silver directory on MinIO
# Coalesce(1) consolidates output into a single partition for easier downstream distribution
items_final.coalesce(1).write \
    .mode("overwrite") \
    .parquet(SILVER_ITEMS_PARQUET_PATH)


# Final confirmation log
print("Done! Silver Order Items table is safely persisted to MinIO as Delta and Parquet.")

Done! Silver Order Items table is safely persisted to MinIO as Delta and Parquet.


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Order Enrichment & Financial Aggregation
Aggregate item-level financial and logistical metrics, enrich the orders table, and persist the updated silver layer while verifying integrity for orphaned records.

In [27]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType

# ==========================================
# 1. READ THE ALREADY SAVED SILVER TABLES (Fast I/O)
# ==========================================
# 1. Read the refined order items table from the Silver layer on MinIO
df_items_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/order_items/")
)

# 2. Read the refined orders table from the Silver layer on MinIO
df_orders_existing = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/orders/")
)

# ==========================================
# 2. CALCULATE FINANCIAL & LOGISTICAL AGGREGATES GROUPED BY ORDER
# ==========================================
# [CORRECTED UNDERSTANDING] This will automatically calculate rows for order_id = "-1" 
# because all orphan items were previously imputed with "-1" inside this notebook!
df_order_aggregates = df_items_silver.groupBy("order_id").agg(
    F.sum("price").cast(DecimalType(10, 2)).alias("total_products_price"),
    F.sum("freight_value").cast(DecimalType(10, 2)).alias("total_freight_value"),
    F.count("order_item_id").cast(IntegerType()).alias("total_items_count"),
    F.countDistinct("seller_id").cast(IntegerType()).alias("seller_count")
).withColumn(
    "total_order_cost",
    (F.col("total_products_price") + F.col("total_freight_value")).cast(DecimalType(10, 2))
).withColumn(
    "is_multi_seller_order",
    F.when(F.col("seller_count") > 1, F.lit(1)).otherwise(F.lit(0)).cast(IntegerType())
)

# ==========================================
# 3. CLEAN EXISTING ORDERS FROM PREVIOUS AGGREGATE COLUMNS
# ==========================================
columns_to_drop = [
    "total_products_price", "total_freight_value", "total_order_cost",
    "total_items_count", "seller_count", "is_multi_seller_order"
]
df_orders_cleaned = df_orders_existing.drop(*columns_to_drop)

# ==========================================
# 4. JOIN & FALLBACK IMPUTATION
# ==========================================
# The join will successfully match "-1" from orders with the calculated "-1" from items!
df_orders_enriched = df_orders_cleaned.join(df_order_aggregates, on="order_id", how="left")

# Fallback defaults will only apply if an order completely lacks items (not the -1 row)
df_orders_enriched = df_orders_enriched \
    .withColumn("total_products_price", F.coalesce(F.col("total_products_price"), F.lit(0.00).cast(DecimalType(10, 2)))) \
    .withColumn("total_freight_value", F.coalesce(F.col("total_freight_value"), F.lit(0.00).cast(DecimalType(10, 2)))) \
    .withColumn("total_order_cost", F.coalesce(F.col("total_order_cost"), F.lit(0.00).cast(DecimalType(10, 2)))) \
    .withColumn("total_items_count", F.coalesce(F.col("total_items_count"), F.lit(0).cast(IntegerType()))) \
    .withColumn("seller_count", F.coalesce(F.col("seller_count"), F.lit(0).cast(IntegerType()))) \
    .withColumn("is_multi_seller_order", F.coalesce(F.col("is_multi_seller_order"), F.lit(0).cast(IntegerType())))

# ==========================================
# 5. OVERWRITE THE SILVER ORDERS TABLE SAFELY
# ==========================================
# ==========================================================================
# FINAL PERSISTENCE: SAVING ENRICHED SILVER ORDERS
# ==========================================================================

# Define the target absolute storage paths on MinIO
SILVER_ORDERS_DELTA_PATH   = "s3a://silver/refined/orders/"
SILVER_ORDERS_PARQUET_PATH = "s3a://silver/refined/orders_parquet/"

# 1. Save the enriched DataFrame as a Delta Table using direct MinIO paths (FIXED: replaced saveAsTable)
df_orders_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_ORDERS_DELTA_PATH)

# Refresh the Delta cache for immediate query capability across the platform
spark.catalog.refreshByPath(SILVER_ORDERS_DELTA_PATH)


# 2. Export as Parquet files to the Silver directory on MinIO (FIXED: updated local path to S3A)
# Coalesce(1) consolidates output into a single partition for easier downstream distribution
df_orders_enriched.coalesce(1).write \
    .mode("overwrite") \
    .parquet(SILVER_ORDERS_PARQUET_PATH)


# Final confirmation log
print("✓ Success: Silver Orders enriched with strict specifications and safely persisted to MinIO!")


✓ Success: Silver Orders enriched with strict specifications and safely persisted to MinIO!
